# 📉 Cuaderno 03 (v2) — Mínimos cuadrados

**Métodos Numéricos · Unidad 4: Ajuste de curvas e interpolación · Tema 4.3**

En el cuaderno 02 construiste polinomios que **pasan exactamente** por cada punto: eso es *interpolar*, y funciona cuando los datos son confiables. En el laboratorio, y en cualquier sensor, cada medición trae error. Si obligas a la curva a tocar todos los puntos, terminas ajustando el polinomio **al ruido** y no a la tendencia.

Este cuaderno cambia la pregunta:

> En vez de "¿qué curva pasa por los puntos?", preguntamos **"¿cuál es la curva que mejor representa la tendencia?"**

La respuesta usa el **criterio de mínimos cuadrados**, el mismo que hay detrás de casi cualquier ajuste de datos en ingeniería.

## 🗺️ Ruta del cuaderno

| § | Tema | Lo que haces | Pregunta |
|---:|---|---|:---:|
| 1 | Interpolar vs. ajustar | Comparas las dos ideas con una figura y un experimento | — |
| 2 | Criterio de mínimos cuadrados | Entiendes *por qué* se eleva al cuadrado el residuo | 1 |
| 3 | Regresión lineal | Deduces las ecuaciones normales y programas `regresion_lineal` | 2 |
| 4 | Regresión cuadrática | Sistema $3\times3$ y el riesgo de sobreajuste | 3 |
| 5 | Error estándar y $r^2$ | Mides la calidad del ajuste | 4 y 5 |
| 6 | Diagnóstico de residuales | Verificas si el modelo tiene la forma correcta | — |

## 📋 Cómo se evalúa

- **5 actividades a mano** (una por método): las resuelves en papel y reportas 4 decimales.
- **5 funciones** que programas **en este cuaderno** y compruebas con `practica(...)`.
- **5 preguntas de opción múltiple**, una por método.

**La calificación va sobre 100 y las preguntas no valen lo mismo:** cada una tiene un peso según su dificultad.

| Pregunta | Tema | Peso |
|:---:|---|:---:|
| 1 | Por qué se eleva al cuadrado el residuo | 15 |
| 2 | De dónde salen las ecuaciones normales | 20 |
| 3 | Elegir entre ajuste lineal y cuadrático | 20 |
| 4 | Grados de libertad del error estándar | 20 |
| 5 | Qué mide (y qué no) el coeficiente de determinación | 25 |
| | **Total** | **100** |

`calificar()` te muestra el puntaje ya ponderado sobre 100. Tienes **2 intentos** de envío: usa el primero cuando `calificar()` te dé todo ✅.

> 💡 **Esta versión no usa GitHub ni Codespaces.** Todo el código se escribe en las celdas del cuaderno. Del repositorio solo se leen el material del curso (`lib/`, `grader/`, `docs/`), que **no** tienes que editar, y las figuras del libro.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
# ---------------------------------------------------------------
# Local: usa el repositorio tal como está en tu computadora.
# Colab: descarga el material del curso en tu Drive (solo lectura).
#        NO necesitas fork ni cuenta de GitHub: tu código vive en
#        el cuaderno, no en el repositorio.
# ---------------------------------------------------------------
import json
import os
import subprocess
import sys

CUADERNO = 'u4_03_minimos_cuadrados'
TAREA_ID = 'U4_03_Minimos_Cuadrados'
REPO_NOMBRE = "U4_Repo"
CURSO = "https://github.com/ITH-MGL-MN/" + REPO_NOMBRE
GUARDADO = os.path.join("CursoMN", "u4_repo.json")   # relativo a MyDrive

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False


def _raiz_local():
    """Sube desde la carpeta actual hasta encontrar lib/robomat.py."""
    candidatos = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        candidatos.append(os.path.dirname(os.path.abspath(nb)))
    candidatos.append(os.getcwd())
    for base in candidatos:
        b = base
        for _ in range(6):
            if os.path.exists(os.path.join(b, "lib", "robomat.py")):
                return b
            padre = os.path.dirname(b)
            if padre == b:
                break
            b = padre
    return None


def _git(*args):
    """Corre git dentro del clon. Devuelve (funcionó, salida)."""
    r = subprocess.run(["git", "-C", REPO] + list(args),
                       capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()


def _traer(enlace):
    """Deja el material en REPO y al día. Devuelve True si lo logró."""
    if os.path.exists(os.path.join(REPO, ".git")):
        ok, _ = _git("fetch", "--depth", "1", "origin", "main")
        if ok:
            _git("reset", "--hard", "origin/main")
            return True
        return False
    r = subprocess.run(["git", "clone", "--depth", "1", enlace + ".git", REPO],
                       capture_output=True, text=True)
    return r.returncode == 0


if not ES_COLAB:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError(
            "No encontré la raíz del repositorio (lib/robomat.py).\n"
            "Abre el cuaderno dentro de U4_Repo o ejecútalo desde esa carpeta.")
    print("Modo        : local (sin repositorio remoto)")
    print("Repositorio :", REPO)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
    REPO = os.path.join(DRIVE, "CursoMN", REPO_NOMBRE)

    print("Material del curso: descargando/actualizando…")
    if not _traer(CURSO):
        # Respaldo: una copia del repositorio guardada en el Drive.
        ruta_json = os.path.join(DRIVE, GUARDADO)
        guardado = {}
        if os.path.exists(ruta_json):
            try:
                with open(ruta_json, encoding="utf-8") as f:
                    guardado = json.load(f)
            except (OSError, json.JSONDecodeError):
                guardado = {}
        enlace = (guardado.get("repo") or "").strip()
        if not enlace:
            enlace = input("Pega el enlace de una copia de U4_Repo: ").strip()
        if not enlace or not _traer(enlace.rstrip("/")):
            raise RuntimeError(
                "No pude descargar el material del curso.\n"
                "Revisa tu conexión o pide el enlace de la copia del repositorio.")
        with open(ruta_json, "w", encoding="utf-8") as f:
            json.dump({"repo": enlace}, f, indent=1)
    print("Repositorio :", REPO)

# Instala dependencias del repositorio solo si falta alguna.
import importlib.util

_modulos_requeridos = ("numpy", "scipy", "sympy", "matplotlib", "yaml", "requests")
_faltantes = [nombre for nombre in _modulos_requeridos
              if importlib.util.find_spec(nombre) is None]
if _faltantes:
    requisitos = os.path.join(REPO, "requirements.txt")
    if not os.path.isfile(requisitos):
        raise FileNotFoundError("Falta requirements.txt en el repositorio.")
    print("Instalando dependencias faltantes:", ", ".join(_faltantes))
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", requisitos],
                   check=True)

# Indica al motor empaquetado dónde está la configuración del repositorio.
os.environ["MN_REPO"] = REPO

# --- Motor de calificación -------------------------------------------
# Se usa el código fuente si está (rama `test`, para depurar); si no, el blob
# empaquetado de grader/, que es lo que recibe el alumno.
_ruta_modulos = os.path.join(REPO, "lib", "mn_motor")
_ruta_blob = os.path.join(REPO, "grader", CUADERNO + ".txt")

_lib = os.path.join(REPO, "lib")
if _lib not in sys.path:
    sys.path.insert(0, _lib)          # ajuste_curvas y, si está, mn_motor

if os.path.isfile(os.path.join(_ruta_modulos, "__init__.py")):
    import mn_motor as _motor
    globals().update({n: getattr(_motor, n) for n in _motor.__all__})
    print("Motor       : módulos de lib/mn_motor (modo profesor)")
else:
    import base64
    import zlib

    if not os.path.exists(_ruta_blob):
        raise FileNotFoundError(
            "No encuentro el motor: falta grader/" + CUADERNO + ".txt\n"
            "Clona el repositorio completo (incluye la carpeta grader/).")
    with open(_ruta_blob, encoding="utf-8") as f:
        codigo = zlib.decompress(base64.b64decode(f.read().strip())).decode("utf-8")
    _MOTOR = {"__name__": "mn_motor_blob", "__file__": "<blob>"}
    exec(compile(codigo, "<blob>", "exec"), _MOTOR)
    globals().update({n: _MOTOR[n] for n in _MOTOR["__all__"]})
    print("Motor       : blob verificado de grader/")

usar_cuaderno(CUADERNO, TAREA_ID)

# --- Preludio estilo MATLAB y herramientas de gráficas ---------------
with open(os.path.join(REPO, "lib", "robomat.py"), encoding="utf-8") as f:
    exec(f.read(), globals())

if REPO not in sys.path:
    sys.path.insert(0, REPO)          # para importar ejercicios/
sys.path.append(os.path.join(REPO, "tools"))
import importlib
import visualizar
importlib.reload(visualizar)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["figure.dpi"] = 110
print("Preludio    : lib/robomat.py · tools/visualizar.py")
print("\nListo. Ejecuta la celda de identificación.")

In [ ]:
#@title 👤 Identificación y semilla reproducible de preguntas
import os

if ES_COLAB:
    import requests
    from google.colab import auth
    import google.auth
    import google.auth.transport.requests

    auth.authenticate_user()
    creds, _ = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    respuesta = requests.get(
        "https://www.googleapis.com/drive/v3/about?fields=user",
        headers={"Authorization": "Bearer " + creds.token},
        timeout=20,
    )
    respuesta.raise_for_status()
    cuenta = respuesta.json().get("user", {})
    email = cuenta.get("emailAddress", "").strip()
    nombre = cuenta.get("displayName", "")
else:
    email = os.environ.get("MN_EMAIL", "").strip()
    if not email:
        email = input("Escribe tu correo institucional @tecnm.mx: ").strip()
    nombre = ""

if not (email and "@" in email and email.lower().endswith(".tecnm.mx")):
    raise ValueError("Usa tu correo institucional terminado en .tecnm.mx.")

MI_TAREA = generar_tarea(email, nombre)
semilla = MI_TAREA["semilla"]
print("Alumno:", email)
print("NC / semilla reproducible:", extraer_nc(email), "/", semilla)
print("Cuaderno:", MI_TAREA["cuaderno"], "·", len(MI_TAREA["preguntas"]), "preguntas")

---

## 1 · ¿Interpolar o ajustar?

### Dos preguntas distintas sobre los mismos puntos

| | **Interpolación** (cuaderno 02) | **Regresión** (este cuaderno) |
|---|---|---|
| Supuesto | Los datos son exactos | Los datos traen ruido experimental |
| Condición | La curva **pasa por** cada punto | La curva **pasa cerca** de todos |
| Parámetros | $n+1$ puntos $\Rightarrow$ polinomio de grado $n$ | $n$ puntos $\Rightarrow$ $m+1$ coeficientes, con $m+1<n$ |
| Efecto del ruido | El polinomio lo copia y oscila | El ruido se promedia y se atenúa |
| Uso típico | Tablas exactas ($\ln 2$, senos) | Mediciones de sensor, experimentos |

La figura **PT5.1** del libro contesta la pregunta con los **mismos cinco puntos**: intenta tres veces ajustar "la mejor" curva y el resultado es muy distinto.

![Figura PT5.1 del libro: tres intentos para ajustar una "mejor" curva con cinco puntos dados: a) regresión por mínimos cuadrados, b) interpolación lineal y c) interpolación curvilínea](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/figPT5-1_ajuste_vs_interpolacion.jpg)

*Figura PT5.1. Tres intentos de ajustar una "mejor" curva a los mismos cinco puntos: a) regresión por mínimos cuadrados, b) interpolación lineal y c) interpolación curvilínea.*

**Qué mirar en la figura, panel por panel**

1. **Panel a), regresión por mínimos cuadrados.** Una recta que pasa **cerca** de los cinco puntos. No toca ninguno exactamente y aun así es la que mejor describe la tendencia: reparte el error entre todos los datos en lugar de concentrarlo en uno.
2. **Panel b), interpolación lineal.** Se une cada punto con el siguiente mediante segmentos rectos. La curva sí pasa por los cinco puntos, pero aparece un zigzag con esquinas: ese zigzag no lo produce la física, lo produce el error de medición.
3. **Panel c), interpolación curvilínea.** Un polinomio suave que pasa por los cinco puntos. Se ve más "bonito" que el zigzag, pero para lograrlo empieza a ondular fuera de la tendencia general: si lo usaras para predecir, el error crecería rápido.
4. **Compáralos en el ojo:** los tres usan los mismos datos; en a) la curva es la más simple y la más parecida a la tendencia, en b) y c) la curva es más complicada y copia el ruido. Esa es toda la diferencia entre **ajustar** e **interpolar**.
5. **Conclusión práctica:** con datos de medición, ajustar; con tablas exactas, interpolar.

> 🧠 **Regla de decisión:** si el dato es exacto, interpola; si el dato es una medición, ajusta.

> 📌 *Si alguna figura no aparece, actualiza el material del repositorio (vuelve a ejecutar la celda ⚙️). Las imágenes viven en `docs/figuras/libro/`.*

Comprobemos la diferencia con un experimento: tomemos seis mediciones con tendencia recta y ruido, y ajustemos **a la vez** una recta (regresión) y un polinomio de grado 5 (interpolación).

In [ ]:
#@title 🔍 Experimento · qué pasa si interpolas datos con ruido
from ajuste_curvas import evaluar_polinomio, minimos_cuadrados

x_demo = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
y_demo = [2.4, 3.1, 4.9, 5.2, 7.1, 6.8]        # tendencia recta + ruido

recta = minimos_cuadrados(x_demo, y_demo, grado=1)   # biblioteca del curso
coef_poli = polyfit(x_demo, y_demo, 5)               # pasa por los 6 puntos

xs = linspace(0.8, 6.6, 200)
plot(xs, evaluar_polinomio(recta["coef"], xs), lw=2, label="ajuste lineal (tendencia)")
plot(xs, polyval(coef_poli, xs), "--", lw=1.2, label="polinomio grado 5 (interpola)")
plot(x_demo, y_demo, "o", ms=9, color="black", label="mediciones")
xlabel("x"), ylabel("y"), grid(alpha=0.3), legend()
title("Interpolar (línea punteada) vs. ajustar (línea sólida)")
show()

sr_poli = sum([(yi - polyval(coef_poli, xi)) ** 2 for xi, yi in zip(x_demo, y_demo)])
print(f"Sr del polinomio que interpola : {sr_poli:.2e}   <- cero, pero no por ser mejor")
print(f"Sr de la recta ajustada        : {recta['Sr']:.4f}")
print(f"Predicción en x = 6.5 · recta      : {evaluar_polinomio(recta['coef'], 6.5):8.4f}")
print(f"Predicción en x = 6.5 · polinomio  : {polyval(coef_poli, 6.5):8.4f}   <- se dispara")

**Lectura del experimento.** El polinomio de grado 5 logra $S_r=0$: pasa exactamente por los seis puntos. Sin embargo su predicción fuera del rango (en $x=6.5$) se dispara, porque para tocar los seis puntos ruidosos tuvo que inventar oscilaciones. La recta tiene $S_r$ mayor y **predice mejor**.

Moraleja: $S_r$ pequeño no es, por sí solo, señal de buen ajuste. Hay que elegir un modelo con **menos parámetros que datos** y verificar la forma de los errores.

---

## 2 · El criterio de mínimos cuadrados

### 2.1 El residuo: cuánto se equivoca el modelo en cada punto

Con $n$ mediciones $(x_1,y_1),\dots,(x_n,y_n)$ y un modelo $y=a_0+a_1x$, para cada dato definimos el **residuo** como la diferencia vertical entre la medición y la curva:

$$e_i \;=\; y_i-\underbrace{\left(a_0+a_1x_i\right)}_{\text{modelo}(x_i)} \tag{17.1}$$

Si $e_i>0$ el modelo quedó **por debajo** del dato; si $e_i<0$, por **arriba**. Es una distancia vertical, no perpendicular: en regresión lo que importa es cuánto se equivoca el modelo en $y$ para una $x$ dada.

El criterio de mínimos cuadrados elige los coeficientes que minimizan la **suma de los residuos al cuadrado**:

$$S_r=\sum_{i=1}^{n}e_i^2=\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i\right)^2 \tag{17.3}$$

### 2.2 ¿Por qué al cuadrado y no otra cosa?

Antes de aceptar (17.3) hay que descartar tres criterios que parecen razonables. La figura 17.2 muestra, en sus tres paneles, exactamente qué falla en cada uno:

![Figura 17.2 del libro: criterios inadecuados para "el mejor ajuste": a) suma de residuos, b) suma de valores absolutos, c) minimax](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-2_criterios.jpg)

*Figura 17.2. Tres criterios para "el mejor ajuste" que son inadecuados para la regresión.*

| Panel | Criterio | Qué muestra la figura | Por qué falla |
|:---:|---|---|---|
| **a** | minimizar $\sum e_i$ | Dos o más puntos y varias rectas que pasan por su "centro" | Los residuos positivos y negativos **se cancelan**: hay infinitas rectas con suma cero |
| **b** | minimizar $\sum\lvert e_i\rvert$ | Cuatro puntos y una **banda** de rectas admisibles | Cualquier recta dentro de la banda empata en el mínimo: **no hay solución única** |
| **c** | minimax, $\min\max\lvert e_i\rvert$ | Un punto alejado del resto | Un solo punto atípico **domina** el ajuste y se ignoran los demás |

**Qué observar en (a).** Las rectas dibujadas pasan todas por la zona media de los puntos; para cualquiera de ellas la suma de residuos con signo vale cero, aunque una esté visiblemente peor que otra. Una medida que da el mismo valor al buen y al mal ajuste no sirve para elegir.

**Qué observar en (b).** El valor absoluto sí evita la cancelación, pero la función es "plana" por tramos: hay toda una familia de rectas empatadas. Necesitamos un criterio con **un único** ganador.

**Qué observar en (c).** El criterio minimax gira la recta solo para bajar el error máximo; un punto mal medido manda sobre los demás. El ajuste deja de representar al conjunto.

Elevar al cuadrado resuelve las tres objeciones a la vez:

1. **No hay cancelación.** $e_i^2\ge 0$ siempre; los errores se acumulan, nunca se restan.
2. **Hay un único mínimo.** $S_r$ es una función **cuadrática** (una parábola, o un paraboloide) de los coeficientes: tiene un solo punto más bajo.
3. **Se puede derivar.** $S_r$ es suave en todas partes, así que el mínimo se localiza con cálculo: $\partial S_r/\partial a_k=0$. Con $\lvert e_i\rvert$ habría puntos donde la derivada no existe.

> ⚠️ El cuadrado también tiene un costo: castiga mucho los errores grandes, así que una **medición atípica** (outlier) pesa demasiado. Si sospechas de un dato, revísalo **antes** de ajustar.

### 2.3 Puente con optimización (capítulo 13 del libro)

En optimización (capítulo 13 del libro) ya localizaste óptimos con una idea que se repite aquí. El método de **interpolación parabólica** toma tres puntos que encierran al extremo, ajusta **exactamente** una parábola que pase por ellos, deriva y encuentra su vértice:

![Figura 13.6 del libro: interpolación parabólica para localizar un óptimo](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig13-6_parabolica.jpg)

*Figura 13.6. La parábola que pasa por tres puntos que rodean al extremo tiene su vértice cerca del óptimo.*

**Qué mirar en la figura 13.6.** Los tres puntos marcados ($x_0<x_1<x_2$) encierran el máximo de la curva; la parábola punteada los une y su vértice —la línea vertical— cae junto al óptimo real. Lo importante del método es que **el extremo de una parábola se obtiene derivando e igualando a cero**, sin tantear punto por punto.

En mínimos cuadrados pasa lo mismo, con otra variable:

| Optimización (cap. 13) | Mínimos cuadrados (cap. 17) |
|---|---|
| Función objetivo $f(x)$ | $S_r(a_0,a_1,\dots,a_m)$ |
| Variable de decisión $x$ | Los **coeficientes** del modelo |
| Extremo: $\dfrac{df}{dx}=0$ | Mínimo: $\dfrac{\partial S_r}{\partial a_k}=0$ para cada $k$ |
| Un punto óptimo | Un **sistema** de $m+1$ ecuaciones |

Como $S_r$ es cuadrática en los coeficientes, sus derivadas son **lineales**: igualarlas a cero produce un **sistema de ecuaciones lineales**, que ya sabes resolver con los métodos de la Unidad 3. Ese es el puente de todo el tema.

Veámoslo con números. Si dejamos fijo $a_0$ en su valor óptimo y solo movemos $a_1$, la función $S_r(a_1)$ es una parábola y su **mínimo es exactamente el coeficiente $a_1$** que buscamos. (La biblioteca del curso ya resolvió el sistema completo; en la sección 3 deducirás de dónde salen esos números.)

In [ ]:
#@title 🎯 Experimento · S_r(a1) es una parábola y su mínimo es la respuesta
from ajuste_curvas import minimos_cuadrados, x_parabolico

x_s = [1.0, 2.0, 3.0, 4.0, 5.0]
y_s = [2.1, 2.9, 4.2, 4.8, 6.1]          # tendencia recta + ruido

coef_ref = minimos_cuadrados(x_s, y_s, grado=1)["coef"]   # biblioteca del curso
a0_fijo = coef_ref[0]                    # se fija a0 para dibujar S_r como función de a1

a1s = linspace(coef_ref[1] - 0.8, coef_ref[1] + 0.8, 121)
sr_vals = [sum([(yi - (a0_fijo + a1 * xi)) ** 2 for xi, yi in zip(x_s, y_s)])
           for a1 in a1s]

plot(a1s, sr_vals, lw=2)
plot(a1s[argmin(sr_vals)], min(sr_vals), "o", ms=10, color="tab:red",
     label="mínimo de la curva")
axvline(coef_ref[1], color="tab:green", ls="--", label="a1 de la biblioteca")
xlabel("$a_1$"), ylabel("$S_r$"), grid(alpha=0.3), legend()
title("$S_r$ respecto a un solo coeficiente: una parábola")
show()

print(f"Mínimo barrido en la malla        : a1 = {a1s[argmin(sr_vals)]:.6f}")
i0, i1, i2 = 10, 60, 110                 # tres puntos de la parábola
print(f"Vértice por interpolación (13.7)  : a1 = {x_parabolico(a1s[i0], a1s[i1], a1s[i2], sr_vals[i0], sr_vals[i1], sr_vals[i2]):.6f}")
print(f"Coeficiente de la biblioteca      : a1 = {coef_ref[1]:.6f}")
print("\nLos tres caminos dan el mismo número: barrer, derivar una parábola (cap. 13)")
print("o resolver las ecuaciones normales (cap. 17). En la sección 3 verás la fórmula.")

### ✏️ Actividad a mano · suma de residuos al cuadrado

Antes de programar, hazlo una vez con lápiz. Ejecuta la celda: te doy **tus** datos (cambian según tu número de control) y un modelo ya definido. Arma la tabla

| $i$ | $x_i$ | $y_i$ | $\hat y_i=a_0+a_1x_i$ | $e_i=y_i-\hat y_i$ | $e_i^2$ |
|---:|---:|---:|---:|---:|---:|

y reporta $S_r=\sum e_i^2$ con **4 decimales**.

In [ ]:
#@title ✏️ Actividad a mano · Suma de residuos al cuadrado
EJ_1 = mano_enunciado("SUMA_RESIDUOS")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Suma de residuos al cuadrado
# Escribe abajo tu S_r con 4 decimales (escribe el número, no una fórmula).
Sr = 0.0
mano_comprueba("SUMA_RESIDUOS", [Sr])

### 💻 Ahora prográmalo en el cuaderno

Completa la función de la celda siguiente. **No hay que ir a GitHub**: escribes aquí, ejecutas y compruebas.

**Contrato**

```
suma_residuos_cuadrados(x, y, coef) -> float
```

- `x`, `y`: listas del mismo tamaño con los datos.
- `coef`: coeficientes en orden **creciente**; `[a0, a1]` representa $a_0+a_1x$.
- Devuelve **un número** (no una lista, no `None`).

**Pistas**

1. El modelo en cada punto es `a0 + a1*x_i`; para no escribirlo a mano usa `enumerate(coef)` y acumula `a * x_i**k`.
2. Suma `(y_i - modelo)**2` sobre todos los datos.
3. Este resultado se reutiliza más adelante: si lo resuelves bien, los demás métodos lo llamarán.

In [ ]:
#@title 💻 Tu función · Suma de residuos al cuadrado
def suma_residuos_cuadrados(x, y, coef):
    """Devuelve Sr = suma de (y_i - modelo(x_i))^2.

    Args:
        x    : lista de abscisas.
        y    : lista de ordenadas medidas.
        coef : coeficientes en orden creciente [a0, a1, a2, ...].

    Returns:
        Sr : número con la suma de los residuos al cuadrado.
    """
    # Pista 1: modelo_i = suma de a_k * x_i**k recorriendo coef con enumerate.
    # Pista 2: acumula (y_i - modelo_i)**2 y devuelve el total.
    raise NotImplementedError("Escribe tu suma_residuos_cuadrados")

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("SUMA_RESIDUOS")

In [ ]:
#@title 📊 Laboratorio · Suma de residuos al cuadrado
# Mira las líneas rojas: cada una es un e_i. Su longitud al cuadrado se suma en S_r.
visualizar.laboratorio("SUMA_RESIDUOS", EJ_1)

In [ ]:
#@title 🧪 Pregunta 1 · Por qué se eleva al cuadrado (vale 15 de 100)
pregunta(1)

---

## 3 · Regresión lineal: las ecuaciones normales

Modelo: $y_i=a_0+a_1x_i+e_i$, con

$$S_r=\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i\right)^2 .$$

### 3.1 Derivar e igualar a cero, paso a paso

**Respecto a $a_0$.** La derivada parcial trata a $a_1$ como constante y usa la regla de la cadena (derivada de algo al cuadrado = $2\cdot$algo$\cdot$derivada de algo):

$$\frac{\partial S_r}{\partial a_0}=\sum_{i=1}^{n}2\left(y_i-a_0-a_1x_i\right)\cdot(-1)=-2\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i\right)=0$$

Dividiendo entre $-2$ y repartiendo la suma en tres términos:

$$\sum y_i-na_0-a_1\sum x_i=0 \quad\Longrightarrow\quad n\,a_0+\left(\sum x_i\right)a_1=\sum y_i \tag{17.4}$$

**Respecto a $a_1$.** Ahora la derivada interna es $-x_i$:

$$\frac{\partial S_r}{\partial a_1}=\sum_{i=1}^{n}2\left(y_i-a_0-a_1x_i\right)\cdot(-x_i)=-2\sum_{i=1}^{n}x_i\left(y_i-a_0-a_1x_i\right)=0$$

$$\left(\sum x_i\right)a_0+\left(\sum x_i^2\right)a_1=\sum x_iy_i \tag{17.5}$$

Dos ecuaciones, dos incógnitas: **son lineales** en $a_0$ y $a_1$ precisamente porque $S_r$ era cuadrática en ellos.

### 3.2 El mismo problema, en forma matricial (Unidad 3)

$$\begin{bmatrix}
n & \sum x_i\\
\sum x_i & \sum x_i^2
\end{bmatrix}
\begin{bmatrix} a_0\\ a_1 \end{bmatrix}
=
\begin{bmatrix} \sum y_i\\ \sum x_i y_i \end{bmatrix}$$

Observa tres cosas:

- La **matriz solo depende de las $x$** y es simétrica; el lado derecho depende de $y$.
- Es un sistema $2\times2$: se resuelve con eliminación de Gauss, LU o `solve`, exactamente como en la Unidad 3.
- La solución en forma cerrada es

$$a_1=\frac{n\sum x_iy_i-\left(\sum x_i\right)\left(\sum y_i\right)}{n\sum x_i^2-\left(\sum x_i\right)^2},\qquad a_0=\bar y-a_1\bar x \tag{17.6–17.7}$$

**Dos comprobaciones gratis.** (1) $a_0=\bar y-a_1\bar x$ dice que la recta **pasa por el punto medio** $(\bar x,\bar y)$. (2) En un ajuste con término constante los residuos suman cero: $\sum e_i=0$.

En código, con la sintaxis del curso:

```python
A = M[[n, Sx], [Sx, Sx2]]
b = M[[Sy], [Sxy]]
mostrar(M[[A, b]])      # la matriz aumentada que resolverías a mano
```

### 3.3 Ejemplo resuelto con los datos de la tabla 17.1

| $x_i$ | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|---|---|---|---|---|---|---|---|
| $y_i$ | 0.5 | 2.5 | 2.0 | 4.0 | 3.5 | 6.0 | 5.5 |

La siguiente celda calcula las sumas con Python (no a mano) y resuelve el sistema. **Primero intenta predecir** los valores de $\sum x_i$ y $\sum x_i^2$; son fáciles de verificar.

In [ ]:
#@title 🧮 Ejemplo 17.1 · sumas y ecuaciones normales
x_ej = [1, 2, 3, 4, 5, 6, 7]
y_ej = [0.5, 2.5, 2.0, 4.0, 3.5, 6.0, 5.5]

n   = len(x_ej)
Sx  = sum(x_ej)
Sy  = sum(y_ej)
Sx2 = sum([x ** 2 for x in x_ej])
Sxy = sum([x * y for x, y in zip(x_ej, y_ej)])

print(f"n = {n}   Sx = {Sx}   Sy = {Sy}   Sx2 = {Sx2}   Sxy = {Sxy}")

A = M[[n, Sx], [Sx, Sx2]]
b = M[[Sy], [Sxy]]
mostrar(M[[A, b]])                    # matriz aumentada (2x3)

sol = np.ravel(solve(A, b))            # el sistema tiene solución única
a0_ej, a1_ej = float(sol[0]), float(sol[1])
print(f"\na0 = {a0_ej:.7f}      a1 = {a1_ej:.7f}")
print(f"Recta: y = {a0_ej:.4f} + {a1_ej:.4f} x")

# Comprobaciones de la seccion 3.2
print(f"\nPasa por (x̄, ȳ): a0 + a1·x̄ = {a0_ej + a1_ej*Sx/n:.4f}   ȳ = {Sy/n:.4f}")
residuos_ej = [yi - (a0_ej + a1_ej*xi) for xi, yi in zip(x_ej, y_ej)]
print(f"Suma de residuos              = {sum(residuos_ej):+.2e}  (~0)")

Con $a_1=0.8392857$ y $a_0=0.0714286$, la recta es $y=0.0714+0.8393x$. Compárala con (17.6)–(17.7): el mismo número, pero obtenido resolviendo el sistema en lugar de memorizando la fórmula.

**Ahora tú**, con tus propios datos:

In [ ]:
#@title ✏️ Actividad a mano · Regresión lineal
EJ_2 = mano_enunciado("REG_LINEAL")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión lineal
# Reporta a0, a1 y Sr, en ese orden, con 4 decimales.
# En papel: completa las sumas, escribe la matriz aumentada y resuelve (Gauss).
# En código, para revisar tu cuenta: A = M[[n, Sx], [Sx, Sx2]] ; b = M[[Sy], [Sxy]].
a0 = 0.0
a1 = 0.0
Sr = 0.0
mano_comprueba("REG_LINEAL", [a0, a1, Sr])

### 💻 Ahora prográmalo en el cuaderno

**Contrato**

```
regresion_lineal(x, y) -> [a0, a1]
```

- Devuelve una **lista de dos números** (no un diccionario, no `None`).
- La pendiente va **segunda**; el orden es el de la recta: $a_0+a_1x$.

**Pistas**

1. Calcula las cinco sumas: `n`, `Sx`, `Sy`, `Sx2`, `Sxy`.
2. Construye `A = M[[n, Sx], [Sx, Sx2]]` y `b = M[[Sy], [Sxy]]`, y resuélvelo con `solve(A, b)` (Unidad 3). También puedes usar (17.6)–(17.7).
3. Comprueba que la recta pasa por $(\bar x,\bar y)$; si falla, revisaste mal el orden de los coeficientes.
4. No dupliques cuentas: el $S_r$ que aparece en la actividad a mano lo obtienes llamando a `suma_residuos_cuadrados(x, y, [a0, a1])`.

In [ ]:
#@title 💻 Tu función · Regresión lineal
def regresion_lineal(x, y):
    """Ajusta y = a0 + a1*x por mínimos cuadrados (ecuaciones normales).

    Args:
        x : lista de abscisas.
        y : lista de ordenadas medidas.

    Returns:
        [a0, a1] : coeficientes del ajuste.
    """
    # Pista 1: n, Sx, Sy, Sx2, Sxy con sum() y comprensiones de lista.
    # Pista 2: A = M[[n, Sx], [Sx, Sx2]] ; b = M[[Sy], [Sxy]] ; solve(A, b).
    # Pista 3: devuelve [float(a0), float(a1)].
    raise NotImplementedError("Escribe tu regresion_lineal")

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("REG_LINEAL")

In [ ]:
#@title 📊 Laboratorio · Regresión lineal
# Izquierda: los datos y tu recta. Derecha: los residuales e_i.
visualizar.laboratorio("REG_LINEAL", EJ_2)

In [ ]:
#@title 🧪 Pregunta 2 · De dónde salen las ecuaciones normales (vale 20 de 100)
pregunta(2)

---

## 4 · Regresión cuadrática y polinomial

Cuando la nube de puntos **se curva**, una recta deja estructura en los errores (sección 6). El siguiente modelo agrega un término:

$$y_i=a_0+a_1x_i+a_2x_i^2+e_i$$

$$S_r=\sum_{i=1}^{n}\left(y_i-a_0-a_1x_i-a_2x_i^2\right)^2 \tag{17.18}$$

Ahora hay **tres** derivadas parciales que igualar a cero ($\partial S_r/\partial a_0$, $\partial S_r/\partial a_1$, $\partial S_r/\partial a_2$), y cada una aporta una ecuación:

$$\begin{bmatrix}
n & \sum x_i & \sum x_i^2\\
\sum x_i & \sum x_i^2 & \sum x_i^3\\
\sum x_i^2 & \sum x_i^3 & \sum x_i^4
\end{bmatrix}
\begin{bmatrix} a_0\\ a_1\\ a_2 \end{bmatrix}
=
\begin{bmatrix}
\sum y_i\\
\sum x_i y_i\\
\sum x_i^2 y_i
\end{bmatrix} \tag{17.19}$$

**Observa el patrón** en lugar de memorizar la matriz:

- La matriz es **simétrica** y cada renglón se corre una potencia: $\sum x^0 \to \sum x^4$.
- El lado derecho lleva $\sum x_i^k y_i$ con la misma potencia que el renglón.
- Para grado $m$ queda un sistema $(m+1)\times(m+1)$ con la misma receta. Por eso la regresión **polinomial** es, otra vez, un problema de álgebra lineal de la Unidad 3.

En el cuaderno:

```python
A = M[[n, Sx, Sx2], [Sx, Sx2, Sx3], [Sx2, Sx3, Sx4]]
b = M[[Sy], [Sxy], [Sx2y]]
mostrar(M[[A, b]])
```

### 4.1 Cuidado: más grado no significa mejor ajuste

- Con $n$ datos y grado $m=n-1$ el sistema tiene solución exacta: $S_r=0$… y ya no estás ajustando, estás **interpolando** (con el problema de la sección 1).
- $S_r$ **nunca aumenta** al subir el grado, porque estás minimizando sobre un conjunto de modelos más grande. Que $S_r$ baje no es evidencia de mejora.
- Lo que sí empeora es la predicción entre datos: aparecen **oscilaciones** (sobreajuste).
- El grado se elige por el **fenómeno físico** y por la forma de los residuales, no maximizando $r^2$. El error estándar $s_{y/x}$ (sección 5) castiga los parámetros extra porque divide entre $n-(m+1)$.

La figura 17.11 muestra un caso donde la parábola sí está justificada:

![Figura 17.11 del libro: ajuste de un polinomio de segundo grado a datos experimentales](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-11_regresion_cuadratica.jpg)

*Figura 17.11. La parábola de mínimos cuadrados sigue la tendencia de los datos del ejemplo 17.5.*

**Qué mirar en la figura 17.11.**

1. Los datos **se curvan**: si trazaras una recta, los primeros puntos quedarían por encima y los últimos por debajo (o al revés). Esa curvatura sistemática es la señal de que falta el término $x^2$.
2. La parábola **no pasa por los puntos**: pasa cerca de todos, repartiendo el error. Compara con la figura del experimento de la sección 1, donde el polinomio de grado 5 sí los tocaba todos y oscilaba.
3. La curva es suave y monótona en el rango medido: no inventa ondulaciones. Eso es señal de un grado **adecuado**, no excesivo.

Comprobemos el sobreajuste con los datos de la tabla 17.4, subiendo el grado poco a poco:

In [ ]:
#@title ⚠️ Experimento · qué pasa al subir el grado
from ajuste_curvas import evaluar_polinomio, minimos_cuadrados

x_libro = [0, 1, 2, 3, 4, 5]
y_libro = [2.1, 7.7, 13.6, 27.2, 40.9, 61.1]     # tabla 17.4

print(f"{'grado':>5} {'Sr':>12} {'s_y/x':>9} {'r2':>9}")
for m in (1, 2, 3, 4):
    r = minimos_cuadrados(x_libro, y_libro, grado=m)
    syx = sqrt(r["Sr"] / (len(x_libro) - (m + 1)))
    print(f"{m:>5} {r['Sr']:>12.5f} {syx:>9.4f} {r['r2']:>9.6f}")

xs = linspace(-0.2, 5.2, 300)
plot(x_libro, y_libro, "o", ms=9, color="black", label="datos")
for m, estilo, color in ((1, "--", "tab:gray"), (2, "-", "tab:blue"), (4, ":", "tab:red")):
    coef = minimos_cuadrados(x_libro, y_libro, grado=m)["coef"]
    plot(xs, evaluar_polinomio(coef, xs), estilo, lw=2, color=color,
         label=f"grado {m}")
xlabel("x"), ylabel("y"), grid(alpha=0.3), legend()
title("Subir el grado baja Sr, pero el grado 4 ya empieza a ondular")
show()

print("\nCon grado 4 (5 coeficientes) queda 6 - 5 = 1 grado de libertad:")
print("Sr casi no baja, pero s_y/x sube porque gasta un dato en cada coeficiente.")

**Lectura de la tabla.** $S_r$ cae de $133.0$ a $3.75$ al pasar de grado 1 a 2, y $s_{y/x}$ baja de $5.77$ a $1.12$: el término $x^2$ sí hacía falta. Pero al subir a grado 3 y 4, $S_r$ apenas mejora ($3.37$ y $3.34$) mientras $s_{y/x}$ **empeora** ($1.30$ y $1.83$), porque cada coeficiente extra gasta un dato más y deja menos grados de libertad. La forma de decidir no es "el grado que minimiza $S_r$" sino "el grado más simple cuyos residuales no muestran estructura".

Ahora sí, con **tus** datos:

In [ ]:
#@title ✏️ Actividad a mano · Regresión cuadrática
EJ_3 = mano_enunciado("REG_CUADRATICA")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Regresión cuadrática
# Reporta a0, a1, a2 y r^2, en ese orden, con 4 decimales.
# En papel: arma el sistema 3x3 de (17.19) con las potencias de x.
# En código: A = M[[n, Sx, Sx2], [Sx, Sx2, Sx3], [Sx2, Sx3, Sx4]] ; b = M[[Sy], [Sxy], [Sx2y]].
# Para r^2 necesitas St = suma de (y_i - ȳ)^2 y Sr = suma de e_i^2.
a0 = 0.0
a1 = 0.0
a2 = 0.0
r2 = 0.0
mano_comprueba("REG_CUADRATICA", [a0, a1, a2, r2])

### 💻 Ahora prográmalo en el cuaderno

**Contrato**

```
regresion_cuadratica(x, y) -> [a0, a1, a2]
```

- Tres números en orden creciente de potencia.
- No devuelvas $S_r$ ni $r^2$: solo los coeficientes.

**Pistas**

1. Necesitas estas sumas: `n, Sx, Sx2, Sx3, Sx4, Sy, Sxy, Sx2y`.
2. Arma `A` y `b` como en el recuadro de arriba y usa `solve(A, b)`.
3. Reutiliza `suma_residuos_cuadrados(x, y, coef)` para el $S_r$ que te piden en la actividad a mano; no repitas la suma.
4. Si obtienes un número gigante o `nan`, revisa que `A` no tenga dos renglones iguales (suele ser una potencia mal copiada).

In [ ]:
#@title 💻 Tu función · Regresión cuadrática
def regresion_cuadratica(x, y):
    """Ajusta y = a0 + a1*x + a2*x^2 por mínimos cuadrados (17.19).

    Args:
        x : lista de abscisas.
        y : lista de ordenadas medidas.

    Returns:
        [a0, a1, a2] : coeficientes del ajuste.
    """
    # Pista 1: Sx, Sx2, Sx3, Sx4, Sy, Sxy, Sx2y.
    # Pista 2: A = M[[n, Sx, Sx2], [Sx, Sx2, Sx3], [Sx2, Sx3, Sx4]] ; b = M[[Sy], [Sxy], [Sx2y]].
    # Pista 3: devuelve [float(a) for a in solve(A, b)].
    raise NotImplementedError("Escribe tu regresion_cuadratica")

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("REG_CUADRATICA")

In [ ]:
#@title 📊 Laboratorio · Regresión cuadrática
visualizar.laboratorio("REG_CUADRATICA", EJ_3)

In [ ]:
#@title 🧪 Pregunta 3 · Elegir entre ajuste lineal y cuadrático (vale 20 de 100)
pregunta(3)

---

## 5 · ¿Qué tan bueno es el ajuste?

Ya tenemos la curva. Falta medirla. Se usan dos números distintos y **complementarios**: uno mide el tamaño típico del error, el otro cuánto se reduce la incertidumbre respecto a no tener modelo.

### 5.1 Error estándar de la estimación $s_{y/x}$

$$s_{y/x}=\sqrt{\frac{S_r}{n-(m+1)}} \tag{17.9, 17.20}$$

- Está en **las mismas unidades que $y$**: "el modelo se equivoca en promedio en $s_{y/x}$ unidades".
- Se divide entre $n-(m+1)$ y no entre $n$ porque cada coeficiente estimado **gasta** un dato: quedan $n-(m+1)$ **grados de libertad**. Para una recta ($m=1$) se divide entre $n-2$.
- Si $n-(m+1)\le 0$ no queda información para medir el error: el modelo tiene tantos parámetros como datos y ya no es un ajuste.
- Comparación útil: la desviación **total** de los datos respecto a su media es $s_y=\sqrt{S_t/(n-1)}$. Si $s_{y/x}<s_y$, el ajuste aporta información; su cociente es el que aparece en $r^2$.

### 5.2 Coeficiente de determinación $r^2$

$$S_t=\sum_{i=1}^{n}\left(y_i-\bar y\right)^2,\qquad
r^2=\frac{S_t-S_r}{S_t}=1-\frac{S_r}{S_t} \tag{17.10}$$

$S_t$ es el error que cometeríamos si **no tuviéramos modelo** y predijéramos siempre con el promedio $\bar y$. $S_r$ es el error del modelo. Entonces:

$$r^2=\frac{\text{error sin modelo}-\text{error con modelo}}{\text{error sin modelo}}$$

es la **fracción de la variabilidad explicada**. Si $r^2=0.95$, el modelo reduce en 95 % el error respecto a usar la media; $r=\sqrt{r^2}$ es el coeficiente de correlación.

> ⚠️ $r^2$ alto **no** prueba que el modelo sea correcto. Puede haber curvatura no explicada y aun así salir $r^2=0.9$. Dos conjuntos de datos distintos pueden dar el mismo $r^2$ con formas completamente diferentes; por eso el diagnóstico de la sección 6 es obligatorio.

> ⚠️ Si $S_t=0$ (todos los $y_i$ iguales) la razón no está definida: no hay variabilidad que explicar.

La figura 17.4 del libro es la imagen mental de estas dos cantidades:

![Figura 17.4 del libro: dispersión alrededor de la media y alrededor de la recta de mejor ajuste](https://raw.githubusercontent.com/ITH-MGL-MN/U4_Repo/main/docs/figuras/libro/fig17-4_dispersion.jpg)

*Figura 17.4. a) Dispersión de los datos alrededor de la media; b) dispersión alrededor de la recta de mejor ajuste. El paso de a) a b) es la mejora que mide $r^2$.*

**Qué mirar en la figura 17.4.**

1. En **(a)** las distancias se miden desde cada punto hasta la **línea horizontal de la media**. Las curvas en forma de campana de la derecha resumen ese ancho: es $S_t$.
2. En **(b)** las distancias se miden hasta la **recta inclinada** y las campanas se **estrechan**. Ese ancho menor es $S_r$.
3. La diferencia de anchos entre (a) y (b) es exactamente $S_t-S_r$, la mejora del modelo; dividida entre $S_t$ da $r^2$.
4. Si ambas campanas tuvieran el mismo ancho, la recta no aportaría nada y $r^2=0$: los datos no tendrían tendencia lineal.

Ahora mide tú: primero el error estándar.

In [ ]:
#@title ✏️ Actividad a mano · Error estándar de la estimación
EJ_4 = mano_enunciado("ERROR_ESTANDAR")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Error estándar de la estimación
# Usa (17.20): divide Sr entre los grados de libertad y saca raíz.
syx = 0.0
mano_comprueba("ERROR_ESTANDAR", [syx])

### 💻 Ahora prográmalo en el cuaderno

**Contrato**

```
error_estandar(Sr, n, m) -> float
```

- `Sr`: suma de residuos al cuadrado.
- `n`: número de datos.
- `m`: **grado del modelo** (recta $\to 1$, parábola $\to 2$). Ojo: no es el número de coeficientes, que vale $m+1$.

**Pistas**

1. Grados de libertad: `gl = n - (m + 1)`.
2. Si `gl <= 0`, lanza `ValueError("no quedan grados de libertad")`: es un error de planteamiento, no de cálculo.
3. Devuelve `sqrt(Sr / gl)`.

In [ ]:
#@title 💻 Tu función · Error estándar de la estimación
def error_estandar(Sr, n, m):
    """Error estándar de la estimación (17.9) y (17.20).

    Args:
        Sr : suma de residuos al cuadrado.
        n  : número de datos.
        m  : grado del modelo ajustado.

    Returns:
        s_y/x : error estándar en las unidades de y.
    """
    # Pista: gl = n - (m + 1); valida gl > 0 y devuelve sqrt(Sr / gl).
    raise NotImplementedError("Escribe tu error_estandar")

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("ERROR_ESTANDAR")

In [ ]:
#@title 📊 Laboratorio · Error estándar de la estimación
visualizar.laboratorio("ERROR_ESTANDAR", EJ_4)

In [ ]:
#@title 🧪 Pregunta 4 · Por qué n - (m+1) (vale 20 de 100)
pregunta(4)

Ahora el coeficiente de determinación, que compara el error del modelo con el error de "no tener modelo".

In [ ]:
#@title ✏️ Actividad a mano · Coeficiente de determinación
EJ_5 = mano_enunciado("COEF_DETERMINACION")

In [ ]:
#@title ✅ Comprueba tus valores a mano · Coeficiente de determinación
# Aplica (17.10) con los dos valores que te doy. Reporta r^2 con 4 decimales.
r2 = 0.0
mano_comprueba("COEF_DETERMINACION", [r2])

### 💻 Ahora prográmalo en el cuaderno

**Contrato**

```
coeficiente_determinacion(Sr, St) -> float
```

- `Sr`: variabilidad que **queda** después del ajuste.
- `St`: variabilidad **total** respecto a la media.
- Devuelve un número entre 0 y 1 (con término constante).

**Pistas**

1. Es una razón de dos sumas: `(St - Sr) / St`.
2. Valida que `St` sea distinto de cero; si no, lanza `ValueError`.
3. No confundas las sumas: $S_r$ se mide respecto al **modelo**, $S_t$ respecto a la **media**.

In [ ]:
#@title 💻 Tu función · Coeficiente de determinación
def coeficiente_determinacion(Sr, St):
    """Coeficiente de determinación r^2 (17.10).

    Args:
        Sr : suma de residuos al cuadrado (respecto al modelo).
        St : suma total de cuadrados (respecto a la media).

    Returns:
        r^2 : fracción de la variabilidad explicada por el modelo.
    """
    # Pista: (St - Sr) / St, validando que St no sea cero.
    raise NotImplementedError("Escribe tu coeficiente_determinacion")

In [ ]:
#@title ▶️ Prueba tu función con un ejemplo
practica("COEF_DETERMINACION")

In [ ]:
#@title 📊 Laboratorio · Coeficiente de determinación
visualizar.laboratorio("COEF_DETERMINACION", EJ_5)

In [ ]:
#@title 🧪 Pregunta 5 · Qué mide (y qué no) r^2 (vale 25 de 100)
pregunta(5)

---

## 6 · Diagnóstico de residuales (laboratorio integrador)

Un $r^2$ alto no basta para aceptar un modelo. El chequeo decisivo es **graficar los residuales** $e_i$ contra $x_i$ (o contra $\hat y_i$) y buscar **estructura**. Si el modelo captura la tendencia, los residuales deben verse como una nube sin forma, centrada en cero y de ancho parecido en todo el rango.

| Patrón en los residuales | Interpretación | Qué hacer |
|---|---|---|
| Nube sin forma, centrada en 0 | El modelo captura la tendencia | Aceptar |
| Arco (U o ∩) | Falta el término $x^2$ | Subir a grado 2 |
| Forma de embudo | La varianza cambia con $x$ | Transformar $y$ (por ejemplo $\log y$) |
| Un punto muy alejado | Posible error de medición | Revisar y justificar, no borrar a ciegas |
| Escalón o tendencia | Falta una variable explicativa | Reformular el modelo |

> 💡 En el cuaderno 02 el residuo era cero por construcción y no servía para nada. Aquí es **la** herramienta de diagnóstico.

Apliquemos todo a los datos del ejemplo 17.5 (tabla 17.4): ajustamos grado 1 y grado 2 con la biblioteca del curso, comparamos las tres medidas y miramos los residuales.

In [ ]:
#@title 📊 Laboratorio integrador · ejemplo 17.5 con datos del libro
from IPython.display import Markdown, display
from ajuste_curvas import evaluar_polinomio, minimos_cuadrados

x_libro = [0, 1, 2, 3, 4, 5]
y_libro = [2.1, 7.7, 13.6, 27.2, 40.9, 61.1]          # tabla 17.4

ajustes = {m: minimos_cuadrados(x_libro, y_libro, grado=m) for m in (1, 2)}

filas = ["| Grado | Coeficientes `[a0, a1, ...]` | $S_r$ | $s_{y/x}$ | $r^2$ |",
         "|---:|:--|--:|--:|--:|"]
for m, r in ajustes.items():
    coef = ", ".join(f"{float(a):.5f}" for a in r["coef"])
    syx = sqrt(r["Sr"] / (len(x_libro) - (m + 1)))
    filas.append(f"| {m} | `[{coef}]` | {r['Sr']:.5f} | {syx:.5f} | {r['r2']:.6f} |")
display(Markdown("### Ejemplo 17.5 · comparación de grados\n\n" + "\n".join(filas)))

xs = linspace(-0.2, 5.2, 300)
fig, (ax_fit, ax_res) = plt.subplots(1, 2, figsize=(11, 4.2))
ax_fit.plot(x_libro, y_libro, "o", ms=9, color="black", label="datos (tabla 17.4)")
for m, r in ajustes.items():
    ax_fit.plot(xs, evaluar_polinomio(r["coef"], xs), lw=2, label=f"grado {m}")
    ax_res.stem(x_libro, r["residuales"], basefmt=" ",
                label=f"grado {m}")
ax_fit.set_title("Ajuste de los datos"); ax_fit.set_xlabel("x"); ax_fit.set_ylabel("y")
ax_res.axhline(0, color="black", lw=0.8)
ax_res.set_title("Residuales $e_i = y_i - \\hat y_i$")
ax_res.set_xlabel("x"); ax_res.set_ylabel("residuo")
for ax in (ax_fit, ax_res):
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print("Interpreta: ¿los residuales del grado 1 se ven como nube o como U?")

**Qué concluir de la figura.** Con grado 1 los residuales **dibujan una U**: positivos en los dos extremos ($+5.8$ y $+6.5$) y negativos en el centro (hasta $-6.0$). Ese patrón no es aleatorio: la recta siempre queda corta en los extremos porque le falta el término $x^2$. Con grado 2 los residuales se reparten a ambos lados del cero sin forma definida y $s_{y/x}$ baja de $5.77$ a $1.12$.

Veamos ahora el caso contrario, el más peligroso: un ajuste con $r^2$ **alto** pero modelo equivocado.

In [ ]:
#@title 🪤 Trampa · r^2 alto con el modelo equivocado
# Datos que siguen y = x^2 en el intervalo [0, 5]: sin ruido, tendencia clara.
x_t = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
y_t = [xi ** 2 for xi in x_t]

recta = minimos_cuadrados(x_t, y_t, grado=1)
print(f"Recta ajustada: y = {recta['coef'][0]:.4f} + {recta['coef'][1]:.4f} x")
print(f"r^2 = {recta['r2']:.4f}  <- se ve 'excelente'")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
xs = linspace(-0.2, 5.2, 200)
ax1.plot(x_t, y_t, "o", ms=9, color="black", label="datos")
ax1.plot(xs, evaluar_polinomio(recta["coef"], xs), lw=2, label="recta ajustada")
ax1.set_title(f"El ajuste parece bueno (r² = {recta['r2']:.3f})")
ax1.set_xlabel("x"); ax1.set_ylabel("y")

ax2.stem(x_t, recta["residuales"], basefmt=" ")
ax2.axhline(0, color="black", lw=0.8)
ax2.set_title("Los residuales revelan el error")
ax2.set_xlabel("x"); ax2.set_ylabel("$e_i$")
ax1.grid(alpha=0.3); ax1.legend()
ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nLa recta explica el {100 * recta['r2']:.0f} % de la variabilidad… y es el modelo equivocado.")
print("Los residuales son positivos en los extremos y negativos en el centro: forma de U.")
print("Un vistazo a la gráfica de residuales lo delata de inmediato.")

### ✅ Antes de enviar

1. Las **5 actividades a mano** revisadas (cada una tiene su celda de comprobación).
2. Las **5 funciones** sin `NotImplementedError` y con `practica(...)` diciendo *"Coincide con la referencia"*.
3. Las **5 preguntas** contestadas en su celda (botón *Guardar respuesta*).
4. Hacer la **celda de verificación final**: resume actividades a mano y calcula tu puntaje **sin enviar**.

**Errores comunes que cuestan puntos**

- Redondear a la mitad del cálculo: reporta 4 decimales **solo al final**.
- Confundir $S_t$ (respecto a la media) con $S_r$ (respecto al modelo).
- Usar $m$ como número de coeficientes en $s_{y/x}$: $m$ es el **grado**; los coeficientes son $m+1$.
- Elegir el grado por $r^2$: siempre sube al agregar términos.
- Reportar $r^2$ sin mirar la gráfica de residuales.

📚 Teoría completa en el repositorio: `docs/metodos/04_minimos_cuadrados_lineal_cuadratico.md` y `docs/metodos/05_comparativa_interpolacion_regresion.md`.

In [ ]:
#@title ✅ 1 · Verificación de los ejercicios a mano
mano_comprueba_todos(["SUMA_RESIDUOS", "REG_LINEAL", "REG_CUADRATICA",
                      "ERROR_ESTANDAR", "COEF_DETERMINACION"])

In [ ]:
#@title 📝 2 · Verificación de todas las respuestas (no envía nada)
# El puntaje ya viene ponderado por dificultad: 100 puntos en total.
puntos, maximo = calificar()

In [ ]:
#@title 📤 3 · Envío de las respuestas
enviar(email)

In [ ]:
#@title 🔎 4 · Verificación de lo que ya está enviado (no envía nada)
consultar_calificacion()